In [9]:
# FACE MASK DETECTION MODEL

from tensorflow.keras.utils import image_dataset_from_directory
import tensorflow as tf
import numpy as np

# Our path to the image folder
data_dir = 'data'

# Image settings/paramters for dataset
batch_size = 32
img_height = 180
img_width = 180

# image_dataset_from_directory() goes into all the folders assigns labels based on folder anmes, resizes them 
# groups them in batches, and creates a TF dataset
train_ds = image_dataset_from_directory(
    data_dir, # Our path to folder, from here it automatically creates labels based on what folder images are in
    validation_split = 0.2, # Split so validation data can monitor perfromacne during training
    subset='training', # Means to give me only the training portion
    seed=123, # Controls radnomness, so we always get the same split when we run
    image_size = (img_height,img_width), # Resizes the images
    batch_size=batch_size # We feed 32 images at a time
)

# Same thing but from the validation set
val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=(img_height,img_width),
    batch_size=batch_size
)

Found 7553 files belonging to 2 classes.
Using 6043 files for training.
Found 7553 files belonging to 2 classes.
Using 1510 files for validation.


In [10]:
# We want to make training faster and more efficeint so here we do three optimizations
# Cache data, shuffle data, prefetch data

AUTOTUNE = tf.data.AUTOTUNE

# cache() stores dataset in memeory after the first time it is loaded, so every epoch can just use memoery to read images (way fasterr)
# shuffle(1000) shuffles the order so its no just Mask Mask Mask No Mask No Mask No Mask, 1000 is just a common number we use
# prefetch(buffer_size=AUTOTUNE), prefetch allows for the next batch to load while the current batch is training
# with AUTOTUNE, TF chooses the buffer_size for you, TF looks at CPU speed, Memory avaliable, and dataset size
train_ds_nm = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

# Same as training but no shuffle since we are not using this for learining
val_ds_nm = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [11]:

# Here we normalize the image pixel values so the neural network can learn more efficiently with smalle rnumbers.
# From 0 to 255, to 0 to 1
from tensorflow.keras import layers

# Rescaling is a keras layer and we make it so it multiplies every pixel by 1/255, and save it as a variable
normalization_layer = layers.Rescaling(1./255)

# map() applies a function to every element in the dataset, so with train_ds_nm we go through each batch and perfrom an operation
# lambda is a anonumous function(small one line functions), in this case x is images and y is labels
# We rescale only x (the images); labels y stay unchanged
normalized_train_ds_nm = train_ds_nm.map(lambda x, y: (normalization_layer(x), y))
normalized_val_ds_nm = val_ds_nm.map(lambda x, y: (normalization_layer(x), y))

# Here we just grab one batch from the dataset so we cna inspect
image_batch, labels_batch = next(iter(normalized_train_ds_nm))

# We take the first image from thr batch and check for the smallest and biggest pixel value
first_image = image_batch[0]
print(np.min(first_image), np.max(first_image))

0.0 1.0


In [12]:
class_names = train_ds.class_names
print(class_names)

['with_mask', 'without_mask']


In [13]:
from keras.applications.vgg16 import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# include_top=False  → removes the original Flatten + 3 Dense layers
# input_shape        → (height, width, channels); must match how images are loaded
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(img_height,img_width,3))

# Freeze the VGG16 base
base_model.trainable = False

# Build the transfer learning model
model_vgg16 = models.Sequential([
    base_model,

    # Takes the average of every feature map, so 3X3 turns into 1 number
    # This then turns the outfut from VGG16 which was 5x5x512 to just 512, which is a 1d vector
    layers.GlobalAveragePooling2D(),


    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),

    layers.Dropout(0.3),

    # Here we take everyhting the NN has learned and make our prediction
    # We use softmax since we have multiple probs, it outputs the provs of our 2 classes
    layers.Dense(2, activation='softmax')
])

# Compile the model
model_vgg16.compile(
    optimizer='adam',

    # We use sprase because are labels are like [0,1,0,1] instead of
#      [1,0]
#      [0,1]
#      [1,0]
#      [0,1]
    loss='sparse_categorical_crossentropy',
    # Three Types of Classification Losses are: Binary Crossentropy, Categorical Crossentropy, and Sprase Categorical Crossentropy


    # Metrics that we use for reporting, not learning
    metrics=['accuracy']
)

# Callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train the model
history_vgg16 = model_vgg16.fit(
    normalized_train_ds_nm,
    validation_data=normalized_val_ds_nm,
    epochs=1,
    callbacks=[early_stopping]
)

# View model structure
model_vgg16.summary()

189/189 ━━━━━━━━━━━━━━━━━━━━ 398s 2s/step - accuracy: 0.8322 - loss: 0.3652 - val_accuracy: 0.9391 - val_loss: 0.1774


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 5, 5, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,912,456 (56.89 MB)

 Trainable params: 65,922 (257.51 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

 Optimizer params: 131,846 (515.03 KB)

In [14]:
# Evaluate model on validation data
val_loss, val_accuracy = model_vgg16.evaluate(normalized_val_ds_nm)

print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

48/48 ━━━━━━━━━━━━━━━━━━━━ 74s 2s/step - accuracy: 0.9391 - loss: 0.1774
Validation Loss: 0.1774304062128067
Validation Accuracy: 0.939072847366333
